[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isrunej/Modul_Kinematik_Robot/blob/main/Modul_03_FK_IK_3DOF.ipynb)

# Modul 3: Forward & Inverse Kinematics — Robot 3-DOF Spatial
## Studi Kasus: Robot Arm 3D di Greenhouse — Memetik dari Baris Tanaman

---

### Tujuan Pembelajaran
1. Memahami mengapa 2D tidak cukup di aplikasi nyata
2. Membangun matriks transformasi homogen 4×4 untuk ruang 3D
3. Menerapkan parameter Denavit-Hartenberg (DH) untuk robot 3-DOF
4. Menghitung FK dan IK robot 3-DOF di Python
5. Memvisualisasikan robot dalam ruang 3D

---

### Mengapa 3D?

Robot greenhouse nyata perlu bergerak dalam **3 dimensi**:
- **Sumbu X**: melintang di antara baris tanaman
- **Sumbu Y**: sepanjang baris tanaman (rel)
- **Sumbu Z**: ketinggian (stroberi ada di berbagai ketinggian)

Robot **3-DOF** yang kita bangun:
- **Joint 1** ($\theta_1$): rotasi horizontal — robot berputar menghadap kiri/kanan
- **Joint 2** ($\theta_2$): rotasi lengan atas — naik/turun
- **Joint 3** ($\theta_3$): rotasi lengan bawah — menekuk/meluruskan

Konfigurasi ini seperti **lengan manusia**: bahu berputar, bahu naik-turun, siku menekuk.

---
## Bagian 1: Matriks Transformasi Homogen 4×4

Di 3D, kita gunakan matriks **4×4**:

$$T = \begin{bmatrix} R_{3\times3} & \mathbf{p}_{3\times1} \\ \mathbf{0}_{1\times3} & 1 \end{bmatrix}$$

- $R$ = matriks rotasi $3\times3$
- $\mathbf{p}$ = vektor posisi $[p_x, p_y, p_z]^T$

### Matriks Rotasi Dasar

$$R_z(\theta) = \begin{bmatrix} c\theta & -s\theta & 0 \\ s\theta & c\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}, \quad R_y(\theta) = \begin{bmatrix} c\theta & 0 & s\theta \\ 0 & 1 & 0 \\ -s\theta & 0 & c\theta \end{bmatrix}, \quad R_x(\theta) = \begin{bmatrix} 1 & 0 & 0 \\ 0 & c\theta & -s\theta \\ 0 & s\theta & c\theta \end{bmatrix}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection

# ============================================
# MATRIKS ROTASI DASAR
# ============================================

def Rz(theta):
    """Rotasi terhadap sumbu Z"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c,-s, 0],
                     [s, c, 0],
                     [0, 0, 1]])

def Ry(theta):
    """Rotasi terhadap sumbu Y"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[ c, 0, s],
                     [ 0, 1, 0],
                     [-s, 0, c]])

def Rx(theta):
    """Rotasi terhadap sumbu X"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1, 0,  0],
                     [0, c, -s],
                     [0, s,  c]])

def T_homogen(R, p):
    """Bangun matriks transformasi homogen 4x4 dari R dan p"""
    T = np.eye(4)
    T[:3, :3] = R
    T[:3,  3] = p
    return T

print("Contoh: Rotasi 45° terhadap sumbu Z")
print(np.round(Rz(np.radians(45)), 4))

---
## Bagian 2: Parameter DH (Denavit-Hartenberg)

DH adalah cara **standar** mendefinisikan geometri robot. Setiap link/joint didefinisikan oleh 4 parameter:

| Parameter | Simbol | Arti |
|-----------|--------|------|
| Link length | $a_i$ | Jarak antar sumbu joint (sepanjang sumbu $x_i$) |
| Link twist | $\alpha_i$ | Sudut antar sumbu $z$ (rotasi sepanjang $x_i$) |
| Link offset | $d_i$ | Jarak sepanjang sumbu $z$ |
| Joint angle | $\theta_i$ | Sudut joint yang digerakkan motor |

### Parameter DH Robot Greenhouse 3-DOF

```
Robot: Rotasi-Bahu-Siku (seperti lengan manusia)

 Joint 1: berputar horizontal (sumbu Z)  
 Joint 2: lengan atas naik-turun (sumbu Y lokal)
 Joint 3: siku menekuk (sumbu Y lokal)
```

| Joint $i$ | $a_i$ | $\alpha_i$ | $d_i$ | $\theta_i$ |
|-----------|--------|------------|--------|------------|
| 1 | 0 | 90° | $d_1$ (tinggi base) | $\theta_1$ |
| 2 | $L_1$ | 0° | 0 | $\theta_2$ |
| 3 | $L_2$ | 0° | 0 | $\theta_3$ |

In [ ]:
# ============================================
# PARAMETER ROBOT 3-DOF GREENHOUSE
# ============================================

d1 = 0.20   # tinggi base dari tanah (meter)
L1 = 0.35   # panjang lengan atas
L2 = 0.30   # panjang lengan bawah

def dh_matriks(a, alpha, d, theta):
    """
    Buat matriks transformasi DH 4x4 untuk satu joint.
    
    T = Rot(z,θ) · Trans(z,d) · Trans(x,a) · Rot(x,α)
    """
    ct = np.cos(theta)
    st = np.sin(theta)
    ca = np.cos(alpha)
    sa = np.sin(alpha)
    
    T = np.array([
        [ct,  -st*ca,  st*sa,  a*ct],
        [st,   ct*ca, -ct*sa,  a*st],
        [ 0,      sa,     ca,     d],
        [ 0,       0,      0,     1]
    ])
    return T

print("Fungsi DH siap!")
print(f"Parameter robot: d1={d1}m, L1={L1}m, L2={L2}m")

---
## Bagian 3: Forward Kinematics 3-DOF

In [ ]:
def forward_kinematics_3dof(theta1_deg, theta2_deg, theta3_deg, d1, L1, L2):
    """
    FK robot 3-DOF dengan parameter DH.
    
    Input: sudut dalam DERAJAT
    Output: dict posisi semua titik dan matriks T akhir
    """
    t1 = np.radians(theta1_deg)
    t2 = np.radians(theta2_deg)
    t3 = np.radians(theta3_deg)
    
    # Matriks DH per joint
    # Joint 1: rotasi terhadap Z, translasi ke atas d1
    # α1 = 90° agar sumbu joint 2 tegak lurus joint 1
    T01 = dh_matriks(a=0,  alpha=np.radians(90), d=d1, theta=t1)
    
    # Joint 2: rotasi lengan atas, panjang L1
    T12 = dh_matriks(a=L1, alpha=0,              d=0,  theta=t2)
    
    # Joint 3: rotasi siku, panjang L2
    T23 = dh_matriks(a=L2, alpha=0,              d=0,  theta=t3)
    
    # Matriks kumulatif
    T02 = T01 @ T12
    T03 = T02 @ T23
    
    # Posisi setiap titik
    p_base   = np.array([0, 0, 0])
    p_joint1 = T01[:3, 3]   # ujung link 0 (setelah joint 1)
    p_joint2 = T02[:3, 3]   # ujung link 1 (setelah joint 2)
    p_ee     = T03[:3, 3]   # end-effector
    
    return {
        'base':     p_base,
        'joint1':   p_joint1,
        'joint2':   p_joint2,
        'ee':       p_ee,
        'T_final':  T03
    }

# Tes FK
hasil = forward_kinematics_3dof(
    theta1_deg=45,
    theta2_deg=30,
    theta3_deg=-60,
    d1=d1, L1=L1, L2=L2
)

print("=== FORWARD KINEMATICS 3-DOF ===")
print(f"Posisi Base    : {np.round(hasil['base'],   4)}")
print(f"Posisi Joint 1 : {np.round(hasil['joint1'], 4)}")
print(f"Posisi Joint 2 : {np.round(hasil['joint2'], 4)}")
print(f"Posisi End-Eff : {np.round(hasil['ee'],     4)}")
print(f"\nEnd-Effector di:")
print(f"  x = {hasil['ee'][0]:.4f} m")
print(f"  y = {hasil['ee'][1]:.4f} m")
print(f"  z = {hasil['ee'][2]:.4f} m")

---
## Bagian 4: Visualisasi 3D

In [ ]:
def visualisasi_robot_3d(theta1_deg, theta2_deg, theta3_deg, d1, L1, L2,
                          posisi_stroberi=None, judul='Robot 3-DOF Greenhouse'):
    """
    Visualisasi 3D robot arm dan target stroberi.
    """
    hasil = forward_kinematics_3dof(theta1_deg, theta2_deg, theta3_deg, d1, L1, L2)
    
    p0 = hasil['base']
    p1 = hasil['joint1']
    p2 = hasil['joint2']
    p3 = hasil['ee']
    
    fig = plt.figure(figsize=(10, 8))
    ax  = fig.add_subplot(111, projection='3d')
    
    # --- Gambar link robot ---
    # Link 0: base ke joint1 (tiang vertikal)
    ax.plot([p0[0],p1[0]], [p0[1],p1[1]], [p0[2],p1[2]],
            'k-', linewidth=6, label='Tiang Base')
    
    # Link 1: joint1 ke joint2 (lengan atas)
    ax.plot([p1[0],p2[0]], [p1[1],p2[1]], [p1[2],p2[2]],
            'b-', linewidth=6, solid_capstyle='round', label='Link 1 (lengan atas)')
    
    # Link 2: joint2 ke ee (lengan bawah)
    ax.plot([p2[0],p3[0]], [p2[1],p3[1]], [p2[2],p3[2]],
            'g-', linewidth=5, solid_capstyle='round', label='Link 2 (lengan bawah)')
    
    # --- Gambar joint ---
    for p, label, warna, ukuran in [
        (p0, 'Base',    'black', 80),
        (p1, 'Joint 1', 'navy',  60),
        (p2, 'Joint 2', 'darkgreen', 55),
        (p3, 'End-Eff', 'red',   100),
    ]:
        ax.scatter(*p, s=ukuran, color=warna, zorder=5)
        ax.text(p[0]+0.02, p[1]+0.02, p[2]+0.02, label, fontsize=8)
    
    # --- Gambar stroberi ---
    if posisi_stroberi:
        xs = [s[0] for s in posisi_stroberi]
        ys = [s[1] for s in posisi_stroberi]
        zs = [s[2] for s in posisi_stroberi]
        ax.scatter(xs, ys, zs, s=150, color='red', marker='*', 
                   zorder=6, label='Stroberi')
    
    # --- Gambar lantai greenhouse ---
    xx, yy = np.meshgrid(np.linspace(-0.1, 0.8, 2), 
                          np.linspace(-0.8, 0.8, 2))
    ax.plot_surface(xx, yy, np.zeros_like(xx), 
                    alpha=0.1, color='brown')
    
    # --- Info kotak ---
    info = (f'θ₁={theta1_deg}°, θ₂={theta2_deg}°, θ₃={theta3_deg}°\n'
            f'End-Effector:\n'
            f'  x={p3[0]:.3f} m\n'
            f'  y={p3[1]:.3f} m\n'
            f'  z={p3[2]:.3f} m')
    ax.text2D(0.02, 0.98, info, transform=ax.transAxes,
              va='top', fontsize=9,
              bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    batas = L1 + L2 + 0.1
    ax.set_xlim(-batas, batas)
    ax.set_ylim(-batas, batas)
    ax.set_zlim(0, batas)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.set_title(judul, fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    return hasil

# Tes visualisasi
stroberi_3d = [(0.35, 0.20, 0.35), (0.25, -0.15, 0.40), (0.30, 0.30, 0.25)]
visualisasi_robot_3d(45, 30, -60, d1, L1, L2, stroberi_3d)

---
## Bagian 5: Inverse Kinematics 3-DOF (Geometrik)

Untuk robot ini, kita pisahkan menjadi dua sub-masalah:

### Sub-masalah 1: Arah Horizontal (θ₁)

$$\theta_1 = \text{atan2}(y_{target},\ x_{target})$$

### Sub-masalah 2: Bidang Vertikal (θ₂, θ₃)

Proyeksikan target ke bidang vertikal (seperti masalah 2-DOF di Modul 1 & 2):

$$r_{xy} = \sqrt{x^2 + y^2} \quad \text{(jarak horizontal)}$$
$$z_{eff} = z_{target} - d_1 \quad \text{(ketinggian relatif dari joint 1)}$$

Lalu selesaikan seperti IK 2-DOF planar dengan target $(r_{xy},\ z_{eff})$.

In [ ]:
def inverse_kinematics_3dof(x, y, z, d1, L1, L2):
    """
    IK geometrik robot 3-DOF greenhouse.
    
    Returns dict dengan solusi 'elbow_down' dan 'elbow_up'
    Sudut dalam DERAJAT.
    """
    # --- Sub-masalah 1: Sudut horizontal ---
    theta1 = np.arctan2(y, x)
    
    # --- Sub-masalah 2: Bidang vertikal ---
    r_xy  = np.sqrt(x**2 + y**2)   # jarak horizontal ke target
    z_eff = z - d1                  # ketinggian relatif dari joint 1
    
    # Jarak 3D dari joint1 ke target
    r = np.sqrt(r_xy**2 + z_eff**2)
    
    # Cek jangkauan
    if r > L1 + L2 + 1e-6:
        print(f"❌ Target ({x:.2f}, {y:.2f}, {z:.2f}) terlalu jauh! r={r:.3f} > {L1+L2}")
        return None
    if r < abs(L1 - L2) - 1e-6:
        print(f"❌ Target ({x:.2f}, {y:.2f}, {z:.2f}) terlalu dekat!")
        return None
    
    # IK 2-DOF di bidang vertikal
    cos_t3 = (r**2 - L1**2 - L2**2) / (2 * L1 * L2)
    cos_t3 = np.clip(cos_t3, -1, 1)
    
    def hitung_t2_t3(theta3):
        # Seperti modul 2, tapi 'x' = r_xy dan 'y' = z_eff
        alpha = np.arctan2(z_eff, r_xy)
        beta  = np.arctan2(L2 * np.sin(theta3), L1 + L2 * np.cos(theta3))
        theta2 = alpha - beta
        return theta2
    
    theta3_down = np.arccos(cos_t3)    # elbow down (siku menekuk ke bawah)
    theta3_up   = -np.arccos(cos_t3)   # elbow up
    
    theta2_down = hitung_t2_t3(theta3_down)
    theta2_up   = hitung_t2_t3(theta3_up)
    
    return {
        'elbow_down': (
            np.degrees(theta1),
            np.degrees(theta2_down),
            np.degrees(theta3_down)
        ),
        'elbow_up': (
            np.degrees(theta1),
            np.degrees(theta2_up),
            np.degrees(theta3_up)
        )
    }

# --- Tes IK dengan posisi stroberi nyata ---
x_t, y_t, z_t = 0.35, 0.20, 0.35  # target stroberi

solusi = inverse_kinematics_3dof(x_t, y_t, z_t, d1, L1, L2)

if solusi:
    print(f"=== IK 3-DOF ===")
    print(f"Target: ({x_t}, {y_t}, {z_t}) meter")
    t1d, t2d, t3d = solusi['elbow_down']
    t1u, t2u, t3u = solusi['elbow_up']
    print(f"\nSolusi Elbow-Down: θ₁={t1d:.2f}°, θ₂={t2d:.2f}°, θ₃={t3d:.2f}°")
    print(f"Solusi Elbow-Up  : θ₁={t1u:.2f}°, θ₂={t2u:.2f}°, θ₃={t3u:.2f}°")

In [ ]:
# --- Verifikasi: gunakan sudut IK untuk FK dan cek posisi ---

def verifikasi_ik(x_target, y_target, z_target, solusi, d1, L1, L2):
    print(f"\n=== VERIFIKASI untuk target ({x_target}, {y_target}, {z_target}) ===")
    for nama, (t1, t2, t3) in solusi.items():
        hasil = forward_kinematics_3dof(t1, t2, t3, d1, L1, L2)
        ee = hasil['ee']
        error = np.sqrt((ee[0]-x_target)**2 + (ee[1]-y_target)**2 + (ee[2]-z_target)**2)
        status = '✅' if error < 1e-4 else '❌'
        print(f"  {nama:12s}: EE=({ee[0]:.4f}, {ee[1]:.4f}, {ee[2]:.4f}) | error={error:.2e} {status}")

verifikasi_ik(x_t, y_t, z_t, solusi, d1, L1, L2)

In [ ]:
# --- Visualisasi dua solusi ---

fig = plt.figure(figsize=(14, 6))

for idx, (nama, (t1, t2, t3)) in enumerate(solusi.items()):
    ax = fig.add_subplot(1, 2, idx+1, projection='3d')
    
    hasil = forward_kinematics_3dof(t1, t2, t3, d1, L1, L2)
    p0 = hasil['base']
    p1 = hasil['joint1']
    p2 = hasil['joint2']
    p3 = hasil['ee']
    
    # Robot links
    ax.plot([p0[0],p1[0]], [p0[1],p1[1]], [p0[2],p1[2]], 'k-', linewidth=5)
    ax.plot([p1[0],p2[0]], [p1[1],p2[1]], [p1[2],p2[2]], 'b-', linewidth=5)
    ax.plot([p2[0],p3[0]], [p2[1],p3[1]], [p2[2],p3[2]], 'g-', linewidth=4)
    
    # Joints
    for p, col, sz in [(p0,'k',80),(p1,'navy',60),(p2,'darkgreen',55),(p3,'red',120)]:
        ax.scatter(*p, s=sz, color=col, zorder=5)
    
    # Target stroberi
    ax.scatter(x_t, y_t, z_t, s=200, color='red', marker='*', 
               zorder=10, label='Target Stroberi')
    
    batas = 0.75
    ax.set_xlim(-batas, batas)
    ax.set_ylim(-batas, batas)
    ax.set_zlim(0, batas)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'{nama.replace("_"," ").title()}\nθ₁={t1:.1f}°, θ₂={t2:.1f}°, θ₃={t3:.1f}°',
                 fontsize=10, fontweight='bold')

plt.suptitle('Dua Solusi IK Robot 3-DOF Greenhouse', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Bagian 6: Simulasi Lengkap — Memetik 4 Stroberi di 3D

In [ ]:
# Daftar stroberi di berbagai posisi (x, y, z) dalam meter
daftar_stroberi_3d = [
    (0.40,  0.15, 0.30),   # baris kanan, dekat
    (0.30, -0.20, 0.42),   # baris kiri, tinggi
    (0.50,  0.25, 0.20),   # baris kanan, rendah
    (0.25,  0.35, 0.38),   # baris depan, sedang
]

print("=== RENCANA PEMETIKAN ROBOT 3-DOF ===")
print(f"{'No':>3} | {'Posisi':^20} | {'θ₁':>8} | {'θ₂':>8} | {'θ₃':>8} | {'Status':^12}")
print("-" * 70)

for i, (x, y, z) in enumerate(daftar_stroberi_3d):
    sol = inverse_kinematics_3dof(x, y, z, d1, L1, L2)
    if sol:
        t1, t2, t3 = sol['elbow_down']
        print(f"{i+1:>3} | ({x:.2f}, {y:.2f}, {z:.2f})   | {t1:>7.1f}° | {t2:>7.1f}° | {t3:>7.1f}° | ✅ Terjangkau")
    else:
        print(f"{i+1:>3} | ({x:.2f}, {y:.2f}, {z:.2f})   | {'':>8} | {'':>8} | {'':>8} | ❌ Di luar jangkauan")

---
## Ringkasan Modul 3

| Konsep | Keterangan |
|--------|------------|
| Matriks DH 4×4 | Cara standar mendefinisikan geometri robot 3D |
| FK 3-DOF | $T_{03} = T_{01} \cdot T_{12} \cdot T_{23}$ |
| IK 3-DOF | Dekomposisi: $\theta_1$ dari arah horizontal, lalu IK 2D di bidang vertikal |
| Dua solusi | Tetap ada elbow-up dan elbow-down |

**Perbandingan 2D vs 3D:**

| Aspek | 2-DOF 2D | 3-DOF 3D |
|-------|----------|----------|
| Output FK | (x, y) | (x, y, z) |
| Matriks Transformasi | 3×3 | 4×4 |
| Input IK | (x, y) | (x, y, z) |
| Jumlah solusi | 2 | 2 (untuk konfigurasi ini) |
| Cek jangkauan | $r = \sqrt{x^2+y^2}$ | $r = \sqrt{r_{xy}^2 + z_{eff}^2}$ |

---

## Selamat!

Anda telah menyelesaikan ketiga modul kinematik robot. Untuk proyek tugas akhir, lihat `tugas/Tugas_Proyek.ipynb`.